#### 1. Install dependencies

In [63]:
%pip install --quiet google-adk requests

#### 2. Imports/configuration

In [112]:
import os

# --- Configuration ---
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = "qwiklabs-gcp-01-06373caf63ce"
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"

#### 3a. Loop-exit tool

The critique agent calls this tool when the refined answer is good enough.

In [126]:
from google.adk.tools.tool_context import ToolContext


def exit_loop(tool_context: ToolContext) -> dict:
    """Signal that the answer is approved and the refinement loop can stop.
    Call this only when the refined answer needs no further improvement.
    Returns:
        An empty dict; the side effect is escalating to end the loop.
    """
    tool_context.actions.escalate = True
    return {}

#### 3b. Loop tracing callbacks

Add callbacks to each sub-agent. `before_agent_callback` to count loop iterations and announce the active agent, `after_agent_callback` used to print that agent's output.

In [127]:
from google.adk.agents.callback_context import CallbackContext

# Simple module-level counter so the trace can show which loop pass we are on.
_loop_state = {"iteration": 0}


def reset_loop_trace():
    """Reset the loop iteration counter before each new question."""
    _loop_state["iteration"] = 0


def _before_agent(callback_context: CallbackContext):
    """before_agent_callback: announce the active agent + loop iteration."""
    name = callback_context.agent_name
    # A new loop iteration begins whenever the search_agent starts a pass.
    if name == "search_agent":
        _loop_state["iteration"] += 1
        print(f"\n{'-' * 70}")
        print(f"LOOP ITERATION {_loop_state['iteration']} of 3")
        print("-" * 70)
    it = _loop_state["iteration"]
    print(f"\n>>> AGENT: {name} [iteration {it}]")
    return None


def _after_agent(callback_context: CallbackContext):
    """after_agent_callback: print the agent's output from session state."""
    name = callback_context.agent_name
    state = callback_context.state
    key = {
        "search_agent": "research",
        "refine_agent": "answer",
        "critique_agent": "critique",
    }.get(name)
    if not key:
        return None
    text = state.get(key)
    if not text:
        return None
    snippet = str(text).strip()
    if len(snippet) > 400:
        snippet = snippet[:400] + " ...[truncated]"
    print(f"    [{name} OUTPUT -> state['{key}']]")
    for line in snippet.splitlines():
        print(f"      {line}")
    return None


#### 4. Search sub-agent

Finds the data/facts needed to answer the student's question using Google Search. Output is saved to state as `research`.

In [128]:
from google.adk.agents import Agent
from google.adk.tools import google_search

search_agent = Agent(
    name="search_agent",
    model="gemini-2.5-flash",
    description=(
        "Finds accurate, up-to-date facts and source material needed to "
        "answer a student's question."
    ),
    instruction=(
        "You are a research assistant for a tutoring system. Look at the "
        "student's question in the conversation. Use the `google_search` "
        "tool to gather the key facts, definitions, formulas, and examples "
        "needed to explain the topic accurately.\n\n"
        "If a critique note from a previous round is present in state under "
        "'{critique?}', let it guide any additional searches you perform.\n\n"
        "Output ONLY the raw research findings as concise bullet points "
        "(facts, definitions, sources). Do NOT write the final explanation — "
        "that is the refine agent's job."
    ),
    tools=[google_search],
    output_key="research",
    before_agent_callback=_before_agent,
    after_agent_callback=_after_agent,
)

#### 5. Refine sub-agent

Writes/rewrites a clear, grade-appropriate explanation using the `research` findings and `critique`, if any. Its output is saved to state as `answer`.

In [129]:
from google.adk.agents import Agent

refine_agent = Agent(
    name="refine_agent",
    model="gemini-2.5-flash",
    description=(
        "Writes and rewrites a clear, grade-appropriate explanation that "
        "teaches the underlying concept to the student."
    ),
    instruction=(
        "You are an expert tutor. Using the research findings below and the "
        "student's question, write a polished, educationally sound answer "
        "that also explains the underlying concepts.\n\n"
        "Research findings:\n{research}\n\n"
        "Previous critique (if any):\n{critique?}\n\n"
        "Guidelines:\n"
        "- Match the difficulty and vocabulary to the student's apparent "
        "grade level (e.g. middle school vs. graduate). If unclear, infer a "
        "reasonable level from the topic and phrasing.\n"
        "- Be clear and concise; use step-by-step reasoning where helpful.\n"
        "- Teach the concept, don't just state the answer.\n"
        "- If a previous critique is provided, revise your answer to address "
        "every suggestion.\n\n"
        "Output ONLY the improved answer text."
    ),
    output_key="answer",
    before_agent_callback=_before_agent,
    after_agent_callback=_after_agent,
)

#### 6. Critique sub-agent

Reviews the refine agent's `answer` for clarity, conciseness, correctness, and grade-level appropriateness. If it is good enough, it calls `exit_loop` to stop the loop. Otherwise it writes concrete improvement suggestions to state as `critique`, which feeds the next iteration.

In [130]:
from google.adk.agents import Agent

critique_agent = Agent(
    name="critique_agent",
    model="gemini-2.5-flash",
    description=(
        "Reviews the tutor's answer and either approves it or gives "
        "constructive suggestions to improve it."
    ),
    instruction=(
        "You are a meticulous teaching-quality reviewer for a tutoring "
        "system. Evaluate the candidate answer below against the student's "
        "original question.\n\n"
        "Candidate answer:\n{answer}\n\n"
        "Check that the answer is:\n"
        "1. Clear and concise.\n"
        "2. Factually correct and complete.\n"
        "3. At the proper level of difficulty for the student (e.g. middle "
        "school vs. graduate) and taught in a style that matches the "
        "age/grade level typical for that subject matter.\n"
        "4. Genuinely teaching the underlying concept, not just giving the "
        "answer.\n\n"
        "DECISION:\n"
        "- If the answer meets ALL criteria and needs no further "
        "improvement, call the `exit_loop` tool and output the single word "
        "'APPROVED'.\n"
        "- Otherwise, do NOT call the tool. Output specific, actionable "
        "suggestions for how the refine agent should improve the answer "
        "(e.g. adjust reading level, fix an error, add an example, tighten "
        "wording). Be concrete."
    ),
    tools=[exit_loop],
    output_key="critique",
    before_agent_callback=_before_agent,
    after_agent_callback=_after_agent,
)

#### 7. Loop agent / answer team

A `LoopAgent` runs the three sub-agents in order — **search → refine → critique** — repeating up to **5** times. The loop ends early if the critique agent calls `exit_loop` (answer approved). The final refined `answer` remains in session state for the greeter.

In [131]:
from google.adk.agents import LoopAgent

answer_team = LoopAgent(
    name="answer_team",
    description=(
        "An answer/refine/critique loop that produces a verified, polished, "
        "grade-appropriate tutoring answer."
    ),
    sub_agents=[search_agent, refine_agent, critique_agent],
    max_iterations=5,
)

/tmp/ipykernel_99/2039820909.py:3: DeprecationWarning: LoopAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  answer_team = LoopAgent(


#### 8. Greeter agent (root)

The single entry point. It greets the student, delegates the actual answering to the `answer_team` loop (exposed as an `AgentTool`), then returns a clean, friendly final response built from the team's verified answer.

In [136]:
from google.adk.agents import Agent
from google.adk.tools.agent_tool import AgentTool

greeter_agent = Agent(
    name="greeter_agent",
    model="gemini-2.5-flash",
    description=(
        "A friendly tutoring front-desk agent that answers student "
        "questions using a verified, refined answer team."
    ),
    instruction=(
        "You are a warm, encouraging tutor and the single point of contact "
        "for the student.\n\n"
        "For every student question or homework/essay request:\n"
        "1. Call the `answer_team` tool exactly once, passing the student's "
        "question. It runs a search -> refine -> critique loop and returns a "
        "verified, grade-appropriate answer.\n"
        "2. Present that answer to the student. Start with ONE short, warm "
        "sentence of greeting, then reproduce the answer_team's returned "
        "answer IN FULL and VERBATIM (keep all formatting, steps, and "
        "examples). You may add one short encouraging sign-off at the end.\n\n"
        "IMPORTANT RULES:\n"
        "- Do NOT say things like 'please wait', 'I'm getting the answer', or "
        "'the system will display it'. The answer is already available from "
        "the tool result — include its full content directly in your reply.\n"
        "- Do NOT summarize, shorten, or omit the answer_team's content.\n"
        "- Do NOT invent your own answer; rely entirely on the answer_team "
        "for the substantive content."
    ),
    tools=[AgentTool(agent=answer_team)],
)


#### 9. Run the greeter agent


In [137]:
import asyncio
import random

from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
from google.genai import errors as genai_errors

APP_NAME = "tutoring_app"
USER_ID = "student_1"
SESSION_ID = "session_1"

session_service = InMemorySessionService()

runner = Runner(
    agent=greeter_agent,
    app_name=APP_NAME,
    session_service=session_service,
)


def _clean(text: str) -> str:
    """Render a model string cleanly.

    Sometimes the greeter re-emits newlines as the literal two-character
    sequence backslash-n instead of real line breaks. Decode those (and
    tabs) back into actual whitespace so the answer prints on multiple
    lines instead of one giant line.
    """
    if not text:
        return text
    return text.replace("\\r\\n", "\n").replace("\\n", "\n").replace("\\t", "\t")


async def _run_once(query: str) -> None:
    """Run the greeter workflow once and print the traced result.

    The detailed loop trace (LOOP ITERATION / >>> AGENT / per-agent OUTPUT)
    is produced by the before/after agent callbacks wired onto the loop
    sub-agents, so it shows up even though the loop runs in a nested runner.
    Here we just frame the request and print the greeter's final answer.
    """
    session = await session_service.get_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID
    )
    if session is None:
        session = await session_service.create_session(
            app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID
        )

    reset_loop_trace()

    content = types.Content(role="user", parts=[types.Part(text=query)])

    print("=" * 70)
    print(f"STUDENT: {query}")
    print("=" * 70)

    last_author = None
    greeter_text = None

    async for event in runner.run_async(
        user_id=USER_ID, session_id=SESSION_ID, new_message=content
    ):
        author = event.author

        # Only the greeter's own top-level handoffs surface here; the loop
        # internals are traced by the callbacks.
        if author != last_author:
            print(f"\n>>> AGENT: {author}")
            last_author = author

        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.function_call:
                    print(f"    [CALL] {author} -> {part.function_call.name}")
                if part.function_response:
                    print(
                        f"    [RESULT] {part.function_response.name} -> {author}"
                    )

        if event.is_final_response() and event.content and event.content.parts:
            text = event.content.parts[0].text
            if text:
                greeter_text = text

    # The verified, refined answer lives in state['answer'].
    session = await session_service.get_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID
    )
    verified_answer = session.state.get("answer") if session else None

    if verified_answer:
        final_text = verified_answer
    else:
        final_text = greeter_text

    total = _loop_state["iteration"]
    print(f"\n{'=' * 70}")
    print(f"TUTOR (final response after {total} loop iteration(s)):")
    print("=" * 70)
    if final_text:
        print(_clean(final_text).strip())
    print("=" * 70)


async def ask_agent(query: str, max_retries: int = 4) -> None:
    """Send a question to the greeter agent with retry/backoff on rate limits.

    This workflow makes many model calls per question (greeter + up to 3
    loop passes of search/refine/critique), which can hit Vertex AI's
    per-minute quota and raise a 429 RESOURCE_EXHAUSTED error. We retry
    with exponential backoff so transient rate limits recover on their own.

    Args:
        query: The natural-language question from the student.
        max_retries: How many times to retry on a 429 before giving up.
    """
    for attempt in range(1, max_retries + 1):
        try:
            await _run_once(query)
            return
        except genai_errors.ClientError as exc:
            is_429 = getattr(exc, "code", None) == 429 or "429" in str(exc)
            if not is_429 or attempt == max_retries:
                raise
            wait = min(2 ** attempt + random.uniform(0, 1), 30)
            print(
                f"\n[RATE-LIMIT] 429 hit (attempt {attempt}/{max_retries}). "
                f"Backing off {wait:.1f}s before retrying...\n"
            )
            await asyncio.sleep(wait)


#### 10. Interact with the tutor

Try questions at different grade levels to see the loop tune the answer.

In [139]:
await ask_agent("I am in first grade, how do I perform addition and subtraction?")

STUDENT: I am in first grade, how do I perform addition and subtraction?

>>> AGENT: greeter_agent
    [CALL] greeter_agent -> answer_team

----------------------------------------------------------------------
LOOP ITERATION 1 of 3
----------------------------------------------------------------------

>>> AGENT: search_agent [iteration 1]
    [search_agent OUTPUT -> state['research']]
      Here's how you can learn about addition and subtraction!
      
      ### What is Addition?
      Addition means putting things together or joining groups to find out how many there are in total. We use a special sign called a "plus sign" (+) to show addition. The answer to an addition problem is called the "sum."
      
      **How to do Addition:**
      
      *   **Using objects:** You can use your toys, blocks, or even your fingers t ...[truncated]

>>> AGENT: refine_agent [iteration 1]
    [refine_agent OUTPUT -> state['answer']]
      Hi there, future math whiz! It's super fun to learn abou

In [140]:
await ask_agent("I'm a graduate student. Explain the intuition behind backpropagation in neural networks.")

STUDENT: I'm a graduate student. Explain the intuition behind backpropagation in neural networks.

>>> AGENT: greeter_agent
    [CALL] greeter_agent -> answer_team

----------------------------------------------------------------------
LOOP ITERATION 1 of 3
----------------------------------------------------------------------

>>> AGENT: search_agent [iteration 1]
    [search_agent OUTPUT -> state['research']]
      Backpropagation is a fundamental algorithm that allows artificial neural networks to learn from data and improve their predictions over time. It's often described as the "heart of neural networks" because it's the mechanism through which these networks adjust their internal parameters to minimize errors.
      
      Here's an intuitive breakdown of how it works:
      
      1.  **Forward Pass: Making a Prediction**
          ...[truncated]

>>> AGENT: refine_agent [iteration 1]
    [refine_agent OUTPUT -> state['answer']]
      Backpropagation is the cornerstone algorith

In [141]:
await ask_agent("I'm in 7th grade. Can you explain photosynthesis for my science homework?")

STUDENT: I'm in 7th grade. Can you explain photosynthesis for my science homework?

>>> AGENT: greeter_agent
    [CALL] greeter_agent -> answer_team

----------------------------------------------------------------------
LOOP ITERATION 1 of 3
----------------------------------------------------------------------

>>> AGENT: search_agent [iteration 1]
    [search_agent OUTPUT -> state['research']]
      *   **What is Photosynthesis?**
          *   Photosynthesis is the process by which green plants, algae, and some microorganisms make their own food.
          *   The word "photosynthesis" comes from two Greek words: "photo," meaning light, and "synthesis," meaning "putting together." So, it means "putting together with light."
          *   Plants use light energy to combine water and carbon dioxide to create fo ...[truncated]

>>> AGENT: refine_agent [iteration 1]
    [refine_agent OUTPUT -> state['answer']]
      Hey there! Photosynthesis might sound like a big word, but it's actual